In [6]:
!pip install optuna
!pip install miditok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 21.0 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.9.0
    Uninstalling typing_extensions-4.9.0:
      Successfully uninstalled typing_extensions-4.9.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 4.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.6 MB/s eta 0:00:0000:0100:01


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
class MIDISlidingWindowDataset(Dataset):
    def __init__(self, token_sequences, seq_len, stride=1):
        """
        Args:
            token_sequences: List of 1D integer lists (one list per MIDI file/track)
            seq_len: The size of the context window (e.g., 128 tokens)
            stride: How many tokens to step forward for the next window
        """
        self.seq_len = seq_len
        self.inputs = []
        self.targets = []

        # Build the overlapping windows for every MIDI sequence
        for seq in token_sequences:
            
            # We need at least seq_len + 1 tokens to make an input/target pair
            if len(seq) <= seq_len:
                continue

            # Slide the window across the sequence
            for i in range(0, len(seq) - seq_len - stride, stride):
                # Input window: [i ... i + seq_len - 1]
                input_chunk = seq[i : i + seq_len]

                # Target window: shifted by 1 token -> [i+1 ... i + seq_len]
                target_chunk = seq[i + stride : i + seq_len + stride]


                self.inputs.append(input_chunk)
                self.targets.append(target_chunk)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        # Convert lists to PyTorch LongTensors (required for CrossEntropyLoss)
        x = torch.tensor(self.inputs[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx], dtype=torch.long)
        # print(len(x), len(y)) #debugging
        return x, y
    
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # Linear layers to project the queries, keys, and values
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.value_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, current_hidden, encoder_outputs):
        # current_hidden: (batch, 1, hidden_dim)
        # encoder_outputs: (batch, seq_len, hidden_dim)
        Q = self.query_proj(current_hidden)
        K = self.key_proj(encoder_outputs)
        V = self.value_proj(encoder_outputs)

        # Scaled dot-product attention
        scores = torch.bmm(Q, K.transpose(1, 2)) / (K.size(-1) ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)

        # Context vector
        context = torch.bmm(attention_weights, V)
        return context, attention_weights


class ComposerMIDI(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):
        super(ComposerMIDI, self).__init__()
        self.hidden_size = hidden_size

        # Map MIDI tokens to dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # The core sequential memory
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)

        # Attention layer to reference themes from the prompt
        self.attention = Attention(hidden_size)

        # Final output projection to next-token probabilities
        self.fc_out = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden=None):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)

        # lstm_out shape: (batch_size, seq_len, hidden_size)
        lstm_out, hidden = self.lstm(embedded, hidden)

        # Apply attention. We use the current step's output to query the sequence
        # (In a real autoregressive loop, encoder_outputs would be cached)
        context, attn_weights = self.attention(lstm_out, lstm_out)

        # Concatenate LSTM output and Attention context
        combined = torch.cat((lstm_out, context), dim=2)

        # Predict next token (Cross-Entropy expects unnormalized logits)
        logits = self.fc_out(combined)
        return logits, hidden

In [8]:
from sklearn.model_selection import train_test_split

def prepare_dataloaders(raw_seqs_path, seq_len=128, batch_size=64, stride=4):
    # 1. Load the raw sequences
    all_sequences = torch.load(raw_seqs_path)
    print(f"Loaded {len(all_sequences)} total sequences.")
    train_seqs, test_seqs = train_test_split(all_sequences, test_size=0.2, random_state=42)
    # 3. Create the Datasets (this applies the sliding window dynamically)
    train_dataset = MIDISlidingWindowDataset(train_seqs, seq_len=seq_len, stride=stride)
    test_dataset = MIDISlidingWindowDataset(test_seqs, seq_len=seq_len, stride=stride)

    print(f"Generated {len(train_dataset)} training windows.")
    print(f"Generated {len(test_dataset)} testing windows.")

    # 4. Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=4,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=True,
        num_workers=4,
        pin_memory=True
    )

    return train_loader, test_loader

# --- Usage in your training loop ---
# train_loader, test_loader = prepare_dataloaders("Data/RawTokens/raw_sequences_my_model.pt")
# train_composer(model, train_loader, epochs=50, vocab_size=vocab_size, lr=1e-3)

In [9]:
import json
# from miditok import REMI
# tokenizer = 
# Read the file normally
with open("Compose10k.json", "r") as f:
    data = json.load(f)
hf_model_data = json.loads(data["_model"])
EXACT_VOCAB_SIZE = len(hf_model_data["model"]["vocab"])
print(f"Success! Vocab size is: {EXACT_VOCAB_SIZE}")
# print(data["_model"]["vocab"])
# EXACT_VOCAB_SIZE = len(data["_model"]["vocab"])

Success! Vocab size is: 10000


In [10]:
from miditok import REMI
import sys
from torch import optim
# tokenizer_path = "Compose10k.json"
# tokenizer = REMI(params=tokenizer_path)
# # tokenizer = 
# EXACT_VOCAB_SIZE = len(tokenizer)
print("hi")
def objective(trial):
    """
    Optuna will run this function multiple times. Every 'trial' it will
    pick a new combination of hyperparameters to test.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)
    sys.stdout.flush()
    # 1. Let Optuna suggest hyperparameters!
    # We constrain nhead so it always perfectly divides d_model (a PyTorch requirement)
    embed_size = trial.suggest_categorical("embed_size", [128, 256, 512])
    hidden_size = trial.suggest_categorical("hidden_size", [256, 512, 1024])
    num_layers = trial.suggest_int("num_layers", 1, 4)
    # num_layers = trial.suggest_int('num_layers', 2, 6)             # Baseline is 2-4
    dropout = trial.suggest_float('dropout', 0.1, 0.4, step=0.1)
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    # strideint = trial.suggest_int("stride", 1, 4)
    strideint = trial.suggest_int('stride', 1, 4)
    # Initialize model with these specific suggestions
    # tokenizer.load("./Compose10k.json")
    vocab_size = EXACT_VOCAB_SIZE
    model = ComposerMIDI(
    vocab_size= EXACT_VOCAB_SIZE,
    embed_size= embed_size,
    hidden_size= hidden_size,
    num_layers= num_layers
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    # Load Data (using a smaller batch size to prevent memory errors during tests)
    # train_loader, val_loader = prepare_dataloaders(
    #     "Predict_FINAL_train_tokens.pt",
    #     "Predict_FINAL_test_tokens.pt",
    #     batch_size=128, stride = strideint
    # )

    train_loader, val_loader = prepare_dataloaders(
        "Predict_FINAL_tokens.pt",
        batch_size=128, stride = strideint
    )
    

    # 2. Train for a few epochs to see if this architecture has potential
    epochs = 4
    best_acc = 0.0
    print("parameters{ num_layers: ",num_layers, "embed_size: ", embed_size , "hidden_size: ", hidden_size , ", dropout:",dropout, ", lr:", lr, "}", "stride:",strideint)
    sys.stdout.flush()
    for epoch in range(epochs):
        model.train()
        count = 0
        for data, target in train_loader:
            count+=1
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            logits, _ = model(data)
            print(data.shape, target.shape, logits.shape)
            logits = logits.view(-1, vocab_size)
            print("logits shape:", logits.shape)
            print("target shape:", target.shape)
            loss = criterion(logits, target.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            if count % 400 == 0:
              print("trial", trial.number, "epoch:",epoch ,"songs:",count)

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                logits, _ = model(data)
                logits = logits.view(-1, vocab_size)
                loss = criterion(logits, target.view(-1))
                val_losses.append(loss.item())

        avg_val_loss = np.mean(val_losses)
        # acc = accuracy_score(all_labels, all_preds)
        # best_acc = max(best_acc, acc)

        # Tell Optuna how we are doing. If the score is terrible, Optuna can "prune" (kill) the trial early.
        print("trial:", trial.number, ", Epoch: ",epoch ,", acc:",acc)
        sys.stdout.flush()
        trial.report(avg_val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return best_acc

def save_visual_reports(study):
    """Saves the experiment results to a CSV table and a PNG graph."""
    os.makedirs("reports", exist_ok=True)

    # 1. Save as a Table (CSV)
    df = study.trials_dataframe()
    # Filter out confusing internal Optuna columns for a clean table
    clean_df = df[['number', 'value', 'params_d_model', 'params_nhead', 'params_num_layers', 'params_dropout', 'params_learning_rate', 'state']]
    clean_df.to_csv("reports/tuning_results.csv", index=False)
    print("-> Saved detailed table to reports/tuning_results.csv")

    # 2. Generate a Bar Graph of the Top 5 Trials
    completed_trials = clean_df[clean_df['state'] == 'COMPLETE']
    top_5 = completed_trials.nlargest(5, 'value')

    plt.figure(figsize=(10, 6))
    bars = plt.bar([f"Trial {i}" for i in top_5['number']], top_5['value'], color='skyblue')
    plt.xlabel('Trial Number')
    plt.ylabel('Validation Accuracy')
    plt.title('Top 5 Transformer Architectures')
    plt.ylim(0, 1.0) # Accuracy goes from 0 to 1

    # Add parameter text to the graph for easy reading
    for bar, (_, row) in zip(bars, top_5.iterrows()):
        text = f"L:{row['params_num_layers']}\nH:{row['params_nhead']}\nD:{row['params_d_model']}"
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.15, text, ha='center', color='black', fontsize=9)

    plt.savefig("reports/top_models_graph.png")
    print("-> Saved visual graph to reports/top_models_graph.png")


hi


In [ ]:
import optuna
print("Starting Architecture Tuning with Optuna...")
# Create an Optuna study. We want to 'maximize' the accuracy.
study = optuna.create_study(
    study_name="model_tuning_Composer",
    direction="maximize",
    storage="sqlite:///optuna.db",
    load_if_exists=True
)
print("optimizing: ")
# Run 10 different architectural trials (increase to 50+ when using real data)
study.optimize(objective, n_trials=20)
print("\n=== TUNING COMPLETE ===")
print(f"Best Accuracy: {study.best_value:.4f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
# Generate our artifacts
save_visual_reports(study)

Starting Architecture Tuning with Optuna...


[I 2026-04-29 09:25:02,061] A new study created in RDB with name: model_tuning_Composer


optimizing: 
cuda
Loaded 48580 total sequences.
